In [15]:
from pyspark.sql import SparkSession
 
# Crear una sesión de Spark
spark = SparkSession.builder \
    .appName("Análisis NYC Taxis") \
    .getOrCreate()

In [18]:
from pyspark.sql.types import StructType, StructField, DoubleType, LongType, StringType
from pyspark.sql.functions import col, lit, coalesce, unix_timestamp, to_timestamp

print("Iniciando procesamiento histórico de Taxis (Capa Silver)...")

# SCHEMA ON READ (Fechas como texto para evitar fallos)
esquema_maestro = StructType([
    # Fechas (Las leemos como STRING para evitar el error BINARY de 2009)
    StructField("tpep_pickup_datetime", StringType(), True),
    StructField("tpep_dropoff_datetime", StringType(), True),
    StructField("lpep_pickup_datetime", StringType(), True),
    StructField("lpep_dropoff_datetime", StringType(), True),
    StructField("pickup_datetime", StringType(), True),
    StructField("dropoff_datetime", StringType(), True),
    StructField("Trip_Pickup_DateTime", StringType(), True),
    StructField("Trip_Dropoff_DateTime", StringType(), True),
    
    # Distancias y Precios
    StructField("trip_distance", DoubleType(), True),
    StructField("total_amount", DoubleType(), True),
    StructField("Total_Amt", DoubleType(), True),
    
    # Zonas
    StructField("PULocationID", LongType(), True),
    StructField("DOLocationID", LongType(), True),
    
    # Coordenadas (Hot Spots Antiguos)
    StructField("pickup_latitude", DoubleType(), True),
    StructField("pickup_longitude", DoubleType(), True),
    StructField("dropoff_latitude", DoubleType(), True),
    StructField("dropoff_longitude", DoubleType(), True),
    StructField("Start_Lat", DoubleType(), True),
    StructField("Start_Lon", DoubleType(), True),
    StructField("End_Lat", DoubleType(), True),
    StructField("End_Lon", DoubleType(), True)
])

# LECTURA
ruta_amarillos = "abfss://nyctaxi-raw@stnycanalysis.dfs.core.windows.net/raw/year=*/month=*/type=yellow/*.parquet"
ruta_verdes = "abfss://nyctaxi-raw@stnycanalysis.dfs.core.windows.net/raw/year=*/month=*/type=green/*.parquet"

df_yellow_raw = spark.read.schema(esquema_maestro).parquet(ruta_amarillos)

try:
    df_green_raw = spark.read.schema(esquema_maestro).parquet(ruta_verdes)
except:
    df_green_raw = spark.createDataFrame([], esquema_maestro)

# COALESCE Y ESTANDARIZACIÓN
def estandarizar_taxis(df_crudo, color_flota):
    return df_crudo.select(
        lit(color_flota).alias("taxi_color"),
        
        # Fechas: Primero unificamos (coalesce) y luego forzamos a Fecha (to_timestamp)
        to_timestamp(coalesce(
            col("tpep_pickup_datetime"), 
            col("lpep_pickup_datetime"), 
            col("pickup_datetime"), 
            col("Trip_Pickup_DateTime")
        )).alias("pickup_datetime"),
        
        to_timestamp(coalesce(
            col("tpep_dropoff_datetime"), 
            col("lpep_dropoff_datetime"), 
            col("dropoff_datetime"), 
            col("Trip_Dropoff_DateTime")
        )).alias("dropoff_datetime"),
        
        # Distancia
        col("trip_distance"),
        
        # Precios
        coalesce(col("total_amount"), col("Total_Amt")).alias("total_amount"),
        
        # Zonas
        col("PULocationID"),
        col("DOLocationID")
    )

df_y_clean = estandarizar_taxis(df_yellow_raw, "Yellow")
df_g_clean = estandarizar_taxis(df_green_raw, "Green")

df_unificado = df_y_clean.unionByName(df_g_clean)

# CÁLCULO DE DURACIÓN Y FILTRO DE CALIDAD
df_unificado = df_unificado.withColumn(
    "duration_min", 
    (unix_timestamp(col("dropoff_datetime")) - unix_timestamp(col("pickup_datetime"))) / 60
)

# Filtramos errores de datos
df_final_filtrado = df_unificado.filter(
    (col("pickup_datetime").isNotNull()) & 
    (col("dropoff_datetime").isNotNull()) &
    (col("trip_distance") > 0) & (col("trip_distance") <= 200) & 
    (col("total_amount") >= 2.50) & 
    (col("duration_min") >= 0.5) & (col("duration_min") <= 300)
)

print("Datos históricos estandarizados y filtrados con éxito.")
df_final_filtrado.show(5)

In [6]:
import requests
from notebookutils import mssparkutils

# DESCARGA DEL DICCIONARIO OFICIAL DE ZONAS

# La URL oficial y estática de la ciudad de Nueva York
url_oficial = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

# La ruta en tu Data Lake donde vivirá para siempre
ruta_destino = f"abfss://nyctaxi-raw@stnycanalysis.dfs.core.windows.net/taxi_zone_lookup.csv"

print("Descargando el archivo original de zonas...")

try:
    # Hacemos la petición a la web
    respuesta = requests.get(url_oficial)
    
    if respuesta.status_code == 200:
        # Escribimos el contenido oficial directamente en el Data Lake
        mssparkutils.fs.put(ruta_destino, respuesta.text, True)
        print("¡Éxito! El archivo oficial (con las 265 zonas) se ha guardado en tu Data Lake.")
        print(f"Ruta: {ruta_destino}")
    else:
        print(f"Error al descargar. Código: {respuesta.status_code}")
except Exception as e:
    print(f"Error de conexión: {e}")

In [5]:
# PERFILADO DE DATOS: AUDITORÍA DE COLUMNAS

# Leer un archivo representativo de cada color (usamos enero 2026 como base)
ruta_amarillos = "abfss://nyctaxi-raw@stnycanalysis.dfs.core.windows.net/raw/year=2026/month=01/type=yellow/yellow_tripdata_2026-01.parquet"
ruta_verdes = "abfss://nyctaxi-raw@stnycanalysis.dfs.core.windows.net/raw/year=2026/month=01/type=green/green_tripdata_2026-01.parquet"

print("Analizando metadatos de los archivos Parquet...")
df_yellow = spark.read.parquet(ruta_amarillos)
df_green = spark.read.parquet(ruta_verdes)

# Extraer las columnas y convertirlas en Sets (Conjuntos) de Python
cols_yellow = set(df_yellow.columns)
cols_green = set(df_green.columns)

# Operaciones de Conjuntos
comunes = cols_yellow.intersection(cols_green)
solo_yellow = cols_yellow.difference(cols_green)
solo_green = cols_green.difference(cols_yellow)

# IMPRIMIR EL INFORME DE COLUMNAS
print("\n" + "="*50)
print("MAPA DE COLUMNAS DEL DATA LAKE")
print("="*50)

print(f"\nCOLUMNAS COMUNES ({len(comunes)} en total):")
print("Estas son las columnas que puedes usar directamente para comparar:")
for col in sorted(comunes):
    print(f"  - {col}")

print(f"\nEXCLUSIVAS TAXI AMARILLO ({len(solo_yellow)} en total):")
for col in sorted(solo_yellow):
    print(f"  - {col}")

print(f"\nEXCLUSIVAS TAXI VERDE ({len(solo_green)} en total):")
for col in sorted(solo_green):
    print(f"  - {col}")

In [3]:
from pyspark.sql.functions import col, lit, count, avg, round, unix_timestamp

# RUTAS DE ENTRADA (Capa RAW)
ruta_amarillos = "abfss://nyctaxi-raw@stnycanalysis.dfs.core.windows.net/raw/year=2026/month=01/type=yellow/yellow_tripdata_2026-01.parquet"
ruta_verdes = "abfss://nyctaxi-raw@stnycanalysis.dfs.core.windows.net/raw/year=2026/month=01/type=green/green_tripdata_2026-01.parquet"

print("Leyendo datos crudos...")
df_yellow = spark.read.parquet(ruta_amarillos)
df_green = spark.read.parquet(ruta_verdes)

# ESTANDARIZACIÓN Y LIMPIEZA
print("Estandarizando esquemas...")

# Usamos unix_timestamp para restar los segundos entre las dos fechas y lo dividimos entre 60
df_y_clean = df_yellow.withColumnRenamed("tpep_pickup_datetime", "pickup_datetime") \
                      .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime") \
                      .withColumn("taxi_color", lit("Yellow")) \
                      .withColumn("duration_min", (unix_timestamp(col("dropoff_datetime")) - unix_timestamp(col("pickup_datetime"))) / 60)

df_g_clean = df_green.withColumnRenamed("lpep_pickup_datetime", "pickup_datetime") \
                     .withColumnRenamed("lpep_dropoff_datetime", "dropoff_datetime") \
                     .withColumn("taxi_color", lit("Green")) \
                     .withColumn("duration_min", (unix_timestamp(col("dropoff_datetime")) - unix_timestamp(col("pickup_datetime"))) / 60)

# UNIÓN
# Seleccionamos las columnas universales
columnas_finales = [
    "taxi_color", "pickup_datetime", "dropoff_datetime", 
    "PULocationID", "DOLocationID", "trip_distance", "total_amount", "duration_min"
]

# Juntamos los amarillos y los verdes
df_unificado = df_y_clean.select(columnas_finales).unionByName(df_g_clean.select(columnas_finales))

# Filtro de calidad (Quitamos viajes de duración 0, precios negativos o viajes de más de 5 horas)
df_unificado = df_unificado.filter(
    (col("trip_distance") > 0) & 
    (col("total_amount") > 0) & 
    (col("duration_min") > 0) & 
    (col("duration_min") < 300) 
)

print("Datos unificados y filtrados con éxito.")

# COMPARATIVA (KPIs)
print("\n" + "="*50)
print("COMPARATIVA DE NEGOCIO: AMARILLOS VS VERDES")
print("="*50)

comparativa = df_unificado.groupBy("taxi_color").agg(
    count("*").alias("Total_Viajes"),
    round(avg("trip_distance"), 2).alias("Distancia_Media_Mi"),
    round(avg("duration_min"), 2).alias("Duracion_Media_Min"),
    round(avg("total_amount"), 2).alias("Ticket_Medio_USD"),
    round(avg(col("total_amount") / col("duration_min")), 2).alias("Rentabilidad_USD_por_Min")
)

comparativa.show()

In [9]:
from pyspark.sql.functions import col, lit, unix_timestamp, round

# RUTAS DE ENTRADA Y SALIDA
BASE_URL = f"abfss://nyctaxi-raw@stnycanalysis.dfs.core.windows.net"

ruta_amarillos = f"{BASE_URL}/raw/year=2026/month=01/type=yellow/*.parquet"
ruta_verdes = f"{BASE_URL}/raw/year=2026/month=01/type=green/*.parquet"
ruta_diccionario = f"{BASE_URL}/taxi_zone_lookup.csv"

# Aquí guardaremos el resultado final (Processed)
#ruta_salida_silver = f"{BASE_URL}/silver/viajes_taxis_2026_01/"

print("Leyendo datos crudos y diccionario oficial...")
df_yellow = spark.read.parquet(ruta_amarillos)
df_green = spark.read.parquet(ruta_verdes)
df_zonas = spark.read.option("header", "true").csv(ruta_diccionario)

# ESTANDARIZACIÓN Y UNIÓN (UNION)
print("Limpiando y unificando amarillos y verdes...")

df_y_clean = df_yellow.withColumn("taxi_color", lit("Yellow")) \
    .withColumn("duration_min", (unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) / 60) \
    .withColumnRenamed("tpep_pickup_datetime", "pickup_datetime") \
    .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime")

df_g_clean = df_green.withColumn("taxi_color", lit("Green")) \
    .withColumn("duration_min", (unix_timestamp(col("lpep_dropoff_datetime")) - unix_timestamp(col("lpep_pickup_datetime"))) / 60) \
    .withColumnRenamed("lpep_pickup_datetime", "pickup_datetime") \
    .withColumnRenamed("lpep_dropoff_datetime", "dropoff_datetime")

columnas_kpis = ["taxi_color", "pickup_datetime", "dropoff_datetime", "PULocationID", "DOLocationID", "trip_distance", "total_amount", "duration_min"]
df_unificado = df_y_clean.select(columnas_kpis).unionByName(df_g_clean.select(columnas_kpis))

# Filtro de calidad para evitar errores matemáticos
df_unificado = df_unificado.filter((col("trip_distance") > 0) & (col("total_amount") > 0) & (col("duration_min") > 0))

# JOIN DOBLE
print("Cruzando IDs con los nombres reales de los barrios...")

# Preparamos el diccionario para el Origen (Pick-Up)
df_origen = df_zonas.select(
    col("LocationID").alias("PU_ID"),
    col("Borough").alias("Origen_Distrito"),
    col("Zone").alias("Origen_Barrio")
)

# Preparamos el diccionario para el Destino (Drop-Off)
df_destino = df_zonas.select(
    col("LocationID").alias("DO_ID"),
    col("Borough").alias("Destino_Distrito"),
    col("Zone").alias("Destino_Barrio")
)

# Hacemos los dos cruces (Left Join) y borramos los IDs numéricos que ya no sirven
df_final = df_unificado \
    .join(df_origen, col("PULocationID") == col("PU_ID"), "left") \
    .join(df_destino, col("DOLocationID") == col("DO_ID"), "left") \
    .drop("PULocationID", "DOLocationID", "PU_ID", "DO_ID")

# GUARDAR EN LA CAPA SILVER
# print("Guardando el Dataframe Enriquecido en la Capa Silver...")
# df_final.write.mode("overwrite").parquet(ruta_salida_silver)

print("\n¡PROCESO COMPLETADO! Tu modelo de datos corporativo está listo en:")
# print(f"-> {ruta_salida_silver}")

# Mostramos una pequeña muestra para que veas la magia
df_final.select("taxi_color", "Origen_Barrio", "Destino_Barrio", "trip_distance", "total_amount").show(5, truncate=False)

In [ ]:
spark.stop()